# Self-history autoregressive anomaly-detection baseline

This mirrors TEP notebook 3.2. Each endogenous process variable is predicted only from its own lagged history, and the residual score uses a diagonal precision matrix estimated from healthy training runs. Alarm settings are selected on faulty validation runs and evaluated on every faulty and healthy held-out test run.

In [1]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from IPython.display import display
from carl.causalai_baselines import run_baseline_comparison

baseline_results = run_baseline_comparison("self_history", PROJECT_ROOT)
print("Self-history baseline fit, validation-tuned, and evaluated on held-out test runs.")

Self-history baseline fit, validation-tuned, and evaluated on held-out test runs.


## 2. Validation and hyperparameter selection

As in TEP, the threshold for each window is the maximum smoothed score in the healthy first phase of faulty validation runs. Window and consecutive alarms maximize trajectory-phase F1, with median detection delay breaking ties.

In [2]:
display(baseline_results["validation_grid"].sort_values(
    ["Trajectory-Phase F1", "Median Anomaly Detection Delay"],
    ascending=[False, True],
).reset_index(drop=True))
display(baseline_results["selected_settings"])

,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Detected Runs,Total Faulty Runs,Median Anomaly Detection Delay
0,7,5,3.407215,0.416229,0.279772,0.0,0.437222,0.797778,0.595556,0.0,0.746518,134,225,35.0
1,5,5,3.605607,0.402670,0.263044,0.0,0.416524,0.795556,0.591111,0.0,0.743017,133,225,34.0
2,6,5,3.495081,0.410433,0.272622,0.0,0.428442,0.795556,0.591111,0.0,0.743017,133,225,34.0
3,8,5,3.441393,0.416346,0.279917,0.0,0.437398,0.788889,0.577778,0.0,0.732394,130,225,35.0
4,7,6,3.407215,0.411237,0.274350,0.0,0.430572,0.786667,0.573333,0.0,0.728814,129,225,36.0
5,6,6,3.495081,0.405175,0.266878,0.0,0.421316,0.782222,0.564444,0.0,0.721591,127,225,35.0
6,8,6,3.441393,0.411702,0.274922,0.0,0.431277,0.782222,0.564444,0.0,0.721591,127,225,36.0
7,7,7,3.407215,0.406552,0.269317,0.0,0.424349,0.780000,0.560000,0.0,0.717949,126,225,37.0
8,8,7,3.441393,0.407251,0.270178,0.0,0.425417,0.780000,0.560000,0.0,0.717949,126,225,37.5
9,5,6,3.605607,0.397016,0.256822,0.0,0.408685,0.775556,0.551111,0.0,0.710602,124,225,34.0


,Baseline,Max Lag,Window Size,Consecutive Alarms,Threshold,PCA Components
0,self_history,2,7,5,3.407215,NaN


## 3. Test on faulty time series

The validation-selected settings are fixed for the full faulty test set.

In [3]:
display(baseline_results["faulty_test_metrics"])
display(baseline_results["faulty_test_by_type"])

,Baseline,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Detected Runs,Total Faulty Runs,Median Anomaly Detection Delay
0,self_history,2,7,5,3.407215,0.44761,0.318489,0.0,0.483112,0.786667,0.573333,0.0,0.728814,129,225,32.0


,Fault Type,Runs,Detected_Runs,Run_Detection_Rate,Median_Detection_Delay
0,hydraulic_coupling_shift,75,2,0.026667,286.5
1,reaction_coupling_shift,75,75,1.000000,21.0
2,thermal_coupling_shift,75,52,0.693333,61.5


## 4. Test on fault-free time series

The same fixed threshold is evaluated on all healthy held-out runs. The comparison table is updated as each of notebooks 3.0, 3.1, and 3.2 is run.

In [4]:
display(baseline_results["healthy_test_metrics"])
display(baseline_results["comparison"])

,Baseline,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase FPR,False Alarm Runs,Total Healthy Runs
0,self_history,2,7,5,3.407215,1.0,0.0,0.0,0.0,0.0,0,75


,Baseline,Max Lag,Window Size,Consecutive Alarms,Threshold,PCA Components,Faulty Test Sample F1,Faulty Test Trajectory-Phase F1,Faulty Test Trajectory-Phase TPR,Faulty Test Median Detection Delay,Healthy Test Sample FPR,Healthy Test Trajectory-Phase FPR
0,full_var,2,8,5,2.983853,NaN,0.592920,0.910412,0.835556,31.5,0.0,0.0
1,pca,2,8,5,1.882515,2.0,0.399929,0.888889,0.800000,56.0,0.0,0.0
2,self_history,2,7,5,3.407215,NaN,0.483112,0.728814,0.573333,32.0,0.0,0.0
